# Decision Learning Tree 

**Author: Rownak Deb Kabya -22400196**


Successful completion of Problem 12.6 carries a 10% bonus for the exam. You need to implement your code in a Jupyter notebook and include a training set of your choice and visualize the learned tree. Groups of up to 4 students can make a joint submission. Submit a HTML export of your notebook via iLearn until July 1, 2025. Your submission must include the names and student IDs of all group members.

## Introduction

This Jupyter notebook implements a Decision Learning Tree without using any external libraries for binary attributes.

#### Decision tree learning
The pseudo-code for the decision tree learning algorithm is as follows:
```python
function DT-LEARNING(examples, attributes, parent examples)
    if empty(examples) then return PLURALITY-VAL(parent examples)
    else if all examples have same classification then return the classification
    else if empty(attributes) then return PLURALITY-VAL(examples)
    else
        A ← argmax
            a∈attributes
        IMPORTANCE(a, examples)
        tree ← a new decision tree with root test A
        for vk ∈ A do
            exs ← {e|e ∈ examples ∧ e.A = vk}
            subtree ← DT-LEARNING(exs, attributes − A, examples)
            add a branch to tree with label (A = vk) and subtree subtree
        end for
        return tree
    end if
end function
```

### Training Set
We will first create a training set to test our decision tree learning algorithm. The training set will consist of binary attributes.

In [75]:
import pandas as pd 

data = [
    {'cloud': 0, 'rain': 0, 'Like': 0, 'temp': 0, 'fun': 0},
    {'cloud': 1, 'rain': 1, 'Like': 1, 'temp': 1, 'fun': 1},
    {'cloud': 1, 'rain': 1, 'Like': 1, 'temp': 1, 'fun': 1},
    {'cloud': 1, 'rain': 1, 'Like': 1, 'temp': 1, 'fun': 1},
    {'cloud': 0, 'rain': 0, 'Like': 0, 'temp': 0, 'fun': 0},
    {'cloud': 1, 'rain': 1, 'Like': 1, 'temp': 1, 'fun': 1},
    {'cloud': 1, 'rain': 1, 'Like': 1, 'temp': 1, 'fun': 1},
    {'cloud': 0, 'rain': 0, 'Like': 0, 'temp': 0, 'fun': 0},
    {'cloud': 1, 'rain': 1, 'Like': 1, 'temp': 1, 'fun': 1},
    {'cloud': 1, 'rain': 1, 'Like': 1, 'temp': 1, 'fun': 1},
    {'cloud': 0, 'rain': 0, 'Like': 0, 'temp': 0, 'fun': 0},
    {'cloud': 1, 'rain': 1, 'Like': 1, 'temp': 1, 'fun': 1},
    {'cloud': 1, 'rain': 1, 'Like': 1, 'temp': 1, 'fun': 1},
    {'cloud': 0, 'rain': 0, 'Like': 0, 'temp': 0, 'fun': 0},
    {'cloud': 1, 'rain': 1, 'Like': 1, 'temp': 1, 'fun': 1},
    {'cloud': 1, 'rain': 1, 'Like': 1, 'temp': 1, 'fun': 1},
    {'cloud': 0, 'rain': 0, 'Like': 0, 'temp': 0, 'fun': 0},
    {'cloud': 1, 'rain': 1, 'Like': 1, 'temp': 1, 'fun': 1},
    {'cloud': 1, 'rain': 1, 'Like': 1, 'temp': 1, 'fun': 1},
    {'cloud': 1, 'rain': 1, 'Like': 1, 'temp': 1, 'fun': 1},
    {'cloud': 1, 'rain': 1, 'Like': 1, 'temp': 1, 'fun': 1},
    {'cloud': 0, 'rain': 0, 'Like': 0, 'temp': 0, 'fun': 0},
    {'cloud': 1, 'rain': 1, 'Like': 1, 'temp': 1, 'fun': 1},
    {'cloud': 1, 'rain': 1, 'Like': 1, 'temp': 1, 'fun': 1},
    {'cloud': 0, 'rain': 0, 'Like': 0, 'temp': 0, 'fun': 0},
    {'cloud': 1, 'rain': 1, 'Like': 1, 'temp': 1, 'fun': 1},
    {'cloud': 1, 'rain': 1, 'Like': 1, 'temp': 1, 'fun': 1},
    {'cloud': 0, 'rain': 0, 'Like': 0, 'temp': 0, 'fun': 0},
    {'cloud': 1, 'rain': 1, 'Like': 1, 'temp': 1, 'fun': 1},
    {'cloud': 1, 'rain': 1, 'Like': 1, 'temp': 1, 'fun': 1},
    {'cloud': 0, 'rain': 0, 'Like': 0, 'temp': 0, 'fun': 0},
    {'cloud': 1, 'rain': 1, 'Like': 1, 'temp': 1, 'fun': 1},
    {'cloud': 1, 'rain': 1, 'Like': 1, 'temp': 1, 'fun': 1}
]

df = pd.DataFrame(data)

df

,cloud,rain,Like,temp,fun
0,0,0,0,0,0
1,1,1,1,1,1
2,1,1,1,1,1
3,1,1,1,1,1
4,0,0,0,0,0
5,1,1,1,1,1
6,1,1,1,1,1
7,0,0,0,0,0
8,1,1,1,1,1
9,1,1,1,1,1


In [76]:
def plurality_value(examples, target_attribute):
    return examples[target_attribute].mode()[0]

In [77]:
import numpy as np
def entropy(examples, target_attribute):
    values, counts = np.unique(examples[target_attribute], return_counts=True)
    probabilities = counts / counts.sum()
    return -np.sum(probabilities * np.log2(probabilities + 1e-9))

In [78]:
def information_gain(examples, attribute, target_attribute):
    total_entropy = entropy(examples, target_attribute)
    values = examples[attribute].unique()
    weighted_entropy = 0
    for v in values:
        subset = examples[examples[attribute] == v]
        weighted_entropy += (len(subset) / len(examples)) * entropy(subset, target_attribute)
    return total_entropy - weighted_entropy

In [79]:
class TreeNode:
    def __init__(self, attribute=None, is_leaf=False, classification=None):
        self.attribute = attribute
        self.is_leaf = is_leaf
        self.classification = classification
        self.children = {}  # value: subtree

In [80]:
def dt_learning(examples, attributes, parent_examples, target_attribute):
    if examples.empty:
        return TreeNode(is_leaf=True, classification=plurality_value(parent_examples, target_attribute))
    elif len(examples[target_attribute].unique()) == 1:
        return TreeNode(is_leaf=True, classification=examples[target_attribute].iloc[0])
    elif len(attributes) == 0:
        return TreeNode(is_leaf=True, classification=plurality_value(examples, target_attribute))
    else:
        # Select attribute with highest information gain
        gains = {a: information_gain(examples, a, target_attribute) for a in attributes}
        A = max(gains, key=gains.get)
        node = TreeNode(attribute=A)
        for v in examples[A].unique():
            subset = examples[examples[A] == v]
            remaining_attributes = [attr for attr in attributes if attr != A]
            child = dt_learning(subset, remaining_attributes, examples, target_attribute)
            node.children[v] = child
        return node

In [81]:
attributes = ['cloud', 'rain', 'Like', 'temp']
target_attribute = 'fun'
tree = dt_learning(df, attributes, df, target_attribute)

In [82]:
from graphviz import Digraph

def render_tree(node, dot=None, parent=None, edge_label=''):
    if dot is None:
        dot = Digraph()
    node_id = str(id(node))
    if node.is_leaf:
        dot.node(node_id, f"Leaf: {node.classification}")
    else:
        dot.node(node_id, f"{node.attribute}")
    if parent is not None:
        dot.edge(parent, node_id, label=str(edge_label))
    for attr_value, child in node.children.items():
        render_tree(child, dot, node_id, edge_label=attr_value)
    return dot

dot = render_tree(tree)
dot.render('decision_tree', format='png', view=True)  # view=True will open the image in a viewer

'decision_tree.png'

In [83]:
def predict(tree, instance):
    while not tree.is_leaf:
        value = instance[tree.attribute]
        if value in tree.children:
            tree = tree.children[value]
        else:
            return None  # Unknown value
    return tree.classification

sample = {'cloud': 1, 'rain': 1, 'Like': 1, 'temp': 1, 'fun': 1}
print(f"Prediction for {sample}: {predict(tree, sample)}")

Prediction for {'cloud': 1, 'rain': 1, 'Like': 1, 'temp': 1, 'fun': 1}: 1


Gtk-Message: 21:37:51.102: Failed to load module "xapp-gtk3-module"
Gtk-Message: 21:37:51.102: Failed to load module "canberra-gtk-module"
[0621/213751.141988:WARNING:chrome/app/chrome_main_linux.cc:82] Read channel stable from /app/extra/CHROME_VERSION_EXTRA


[0621/213751.243179:WARNING:chrome/app/chrome_main_linux.cc:82] Read channel stable from /app/extra/CHROME_VERSION_EXTRA
Opening in existing browser session.
